In [0]:

delta_table_path_bronze = "/Volumes/workspace/default/tables/cardio_bronze"

df_bronze = spark.read.format("delta").load(delta_table_path_bronze)

df_bronze.printSchema()
df_bronze.show(5)


root
 |-- id: integer (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: integer (nullable = true)
 |-- height: integer (nullable = true)
 |-- weight: double (nullable = true)
 |-- ap_hi: integer (nullable = true)
 |-- ap_lo: integer (nullable = true)
 |-- cholesterol: integer (nullable = true)
 |-- gluc: integer (nullable = true)
 |-- smoke: integer (nullable = true)
 |-- alco: integer (nullable = true)
 |-- active: integer (nullable = true)
 |-- cardio: integer (nullable = true)

+---+-----+------+------+------+-----+-----+-----------+----+-----+----+------+------+
| id|  age|gender|height|weight|ap_hi|ap_lo|cholesterol|gluc|smoke|alco|active|cardio|
+---+-----+------+------+------+-----+-----+-----------+----+-----+----+------+------+
|  0|18393|     2|   168|  62.0|  110|   80|          1|   1|    0|   0|     1|     0|
|  1|20228|     1|   156|  85.0|  140|   90|          3|   1|    0|   0|     1|     1|
|  2|18857|     1|   165|  64.0|  130|   70|          3|   1| 

In [0]:
from pyspark.sql.functions import col, round

df_silver = df_bronze.withColumn("age_years", round(col("age") / 365.25, 0).cast("integer"))

df_silver.select("age", "age_years").show(5)


+-----+---------+
|  age|age_years|
+-----+---------+
|18393|       50|
|20228|       55|
|18857|       52|
|17623|       48|
|17474|       48|
+-----+---------+
only showing top 5 rows


In [0]:
from pyspark.sql.functions import when

df_silver = df_silver.withColumn("gender_mapped", \
    when(col("gender") == 1, "Feminino") \
    .when(col("gender") == 2, "Masculino") \
    .otherwise("Desconhecido")
)

df_silver = df_silver.withColumn("cholesterol_mapped", \
    when(col("cholesterol") == 1, "Normal") \
    .when(col("cholesterol") == 2, "Acima do Normal") \
    .when(col("cholesterol") == 3, "Bem Acima do Normal") \
    .otherwise("Desconhecido")
)

df_silver = df_silver.withColumn("gluc_mapped", \
    when(col("gluc") == 1, "Normal") \
    .when(col("gluc") == 2, "Acima do Normal") \
    .when(col("gluc") == 3, "Bem Acima do Normal") \
    .otherwise("Desconhecido")
)

df_silver.select("gender", "gender_mapped", "cholesterol", "cholesterol_mapped", "gluc", "gluc_mapped").show(5)


+------+-------------+-----------+-------------------+----+-----------+
|gender|gender_mapped|cholesterol| cholesterol_mapped|gluc|gluc_mapped|
+------+-------------+-----------+-------------------+----+-----------+
|     2|    Masculino|          1|             Normal|   1|     Normal|
|     1|     Feminino|          3|Bem Acima do Normal|   1|     Normal|
|     1|     Feminino|          3|Bem Acima do Normal|   1|     Normal|
|     2|    Masculino|          1|             Normal|   1|     Normal|
|     1|     Feminino|          1|             Normal|   1|     Normal|
+------+-------------+-----------+-------------------+----+-----------+
only showing top 5 rows


In [0]:

df_silver = df_silver.filter(
    (col("ap_lo") <= col("ap_hi")) &  # Diastólica não pode ser maior que sistólica
    (col("ap_hi") >= 70) & (col("ap_hi") <= 250) & # Valores razoáveis para sistólica
    (col("ap_lo") >= 40) & (col("ap_lo") <= 180)    # Valores razoáveis para diastólica
)

print(f"Número de registros após a limpeza de pressão arterial: {df_silver.count()}")


Número de registros após a limpeza de pressão arterial: 68672


In [0]:
# Remover duplicatas, considerando todas as colunas exceto 'id' se ele for apenas um identificador
# Se 'id' for único por natureza, podemos remover duplicatas baseadas em todas as outras colunas
# Ou, se houver chance de 'id' ser duplicado com dados diferentes, podemos remover duplicatas em todas as colunas
# Para este dataset, vamos assumir que 'id' é um identificador único e remover duplicatas baseadas em todas as colunas.

df_silver = df_silver.dropDuplicates()

print(f"Número de registros após a remoção de duplicatas: {df_silver.count()}")


Número de registros após a remoção de duplicatas: 68672


In [0]:
# Caminho para salvar a tabela Delta Lake na camada Silver
delta_table_path_silver = "/Volumes/workspace/default/tables/cardio_silver"

# Salvar o DataFrame transformado em formato Delta Lake
df_silver.write \
  .format("delta") \
  .mode("overwrite") \
  .save(delta_table_path_silver)

print(f"Dados transformados salvos com sucesso em formato Delta Lake em: {delta_table_path_silver}")

df_silver_read = spark.read.format("delta").load(delta_table_path_silver)
df_silver_read.show(5)


Dados transformados salvos com sucesso em formato Delta Lake em: /Volumes/workspace/default/tables/cardio_silver
+----+-----+------+------+------+-----+-----+-----------+----+-----+----+------+------+---------+-------------+-------------------+-----------+
|  id|  age|gender|height|weight|ap_hi|ap_lo|cholesterol|gluc|smoke|alco|active|cardio|age_years|gender_mapped| cholesterol_mapped|gluc_mapped|
+----+-----+------+------+------+-----+-----+-----------+----+-----+----+------+------+---------+-------------+-------------------+-----------+
|  53|18126|     1|   165|  70.0|  140|   90|          1|   1|    0|   0|     1|     1|       50|     Feminino|             Normal|     Normal|
| 749|21398|     1|   170|  88.0|  140|   90|          1|   1|    0|   0|     1|     0|       59|     Feminino|             Normal|     Normal|
|1764|22727|     1|   166|  76.0|  160|   90|          3|   1|    0|   0|     1|     1|       62|     Feminino|Bem Acima do Normal|     Normal|
|5720|18880|     1|   1